# Day 1 — Train / Validation / Test Splits

## Overview

This task focuses on building a proper three-way data split using Training, Validation, and Test sets.

The main goal is to understand the role of each dataset, use the validation set for model development and hyperparameter tuning, and keep the test set completely untouched until the final evaluation.


## Importing the Required Libraries

The following libraries are imported for data loading, preprocessing, data splitting, building machine learning pipelines, and training classification models.


In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier



## Loading the Dataset

The Stroke Prediction dataset is loaded using Pandas.

The dataset was previously explored and analyzed during **Week 3**, where exploratory data analysis (EDA) was performed to understand the dataset, examine feature distributions, identify missing values, and investigate the target variable.

The preprocessing and modeling steps in this task build on that previous analysis.


In [3]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## Defining Features and Target

The target variable is `stroke`, which represents whether a patient experienced a stroke.

The `id` column was removed because it is only an identifier and does not provide meaningful information for predicting stroke.

The remaining columns are used as input features `X`, while `stroke` is stored as the target variable `y`.


In [4]:
X = df.drop(columns=["stroke", "id"])
y = df["stroke"]

## Creating the Test Set

The first step is to hold out **20% of the dataset as the final test set**.

This test set will remain untouched during model development and hyperparameter tuning. It will only be used once at the end to obtain the final performance estimate.

The remaining 80% is stored in `X_temp` and `y_temp` and will later be divided into training and validation sets.


In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Creating the Training and Validation Sets

The remaining 80% of the data is split into:

* **75% training data**, which represents 60% of the original dataset.
* **25% validation data**, which represents 20% of the original dataset.

Therefore, the final split is approximately **60% Training / 20% Validation / 20% Test**.

The `stratify` parameter is used to preserve the distribution of the target classes across the datasets.


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

## Identifying Numerical and Categorical Features

The input features are divided into two groups based on their data types.

Numerical features will be processed using numerical preprocessing techniques, while categorical features will be encoded into numerical representations before being passed to the model.


In [9]:
categorical_features = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

numerical_features = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi"
]

## Building the Preprocessing Pipeline

A `ColumnTransformer` is used to apply different preprocessing steps to numerical and categorical features.

For numerical features:

* Missing values are handled using median imputation.
* Features are standardized using `StandardScaler`.

For categorical features:

* Missing values are replaced using the most frequent value.
* Categorical values are converted into numerical features using `OneHotEncoder`.

The preprocessing steps are fitted only on the training data when the model pipeline is trained. This helps prevent data leakage from the validation and test sets.


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numerical_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

## Hyperparameter Tuning

A Random Forest classifier is used for the classification task.

The `n_estimators` hyperparameter is tuned using the validation set. This parameter controls the number of decision trees in the Random Forest.

Several values are tested while keeping the training, validation, and test sets fixed.

The test set is not used during this tuning process.


In [19]:
for n in [50, 100, 200, 300]:

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=n,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    val_score = model.score(X_val, y_val)

    print(f"n_estimators = {n}, Validation Score = {val_score:.4f}")

n_estimators = 50, Validation Score = 0.9511
n_estimators = 100, Validation Score = 0.9511
n_estimators = 200, Validation Score = 0.9511
n_estimators = 300, Validation Score = 0.9511


## Validation Results

The validation results show that changing the `n_estimators` value from 50 to 300 did not change the validation accuracy.

This can happen when the additional trees do not provide enough new information to change the model's predictions on the validation set. In this case, the model reached the same accuracy across all tested values.

Since all tested values achieved the same validation score, `n_estimators = 100` was selected for the final model.


## Validation Results

The validation results show that changing the `n_estimators` value from 50 to 300 did not change the validation accuracy.

This can happen when the additional trees do not provide enough new information to change the model's predictions on the validation set. In this case, all tested values achieved the same validation accuracy of **95.11%**.

Since all tested values performed equally on the validation set, `n_estimators = 100` was selected for the final model as a reasonable balance between model complexity and training time. This choice was based only on the validation results, and the test set was not used during the selection process.


In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

test_score = final_model.score(X_test, y_test)

print(f"Final Test Score: {test_score:.4f}")

Final Test Score: 0.9462


## Final Test Evaluation

The final Random Forest model achieved:

* **Validation Accuracy:** 95.11%
* **Final Test Accuracy:** 94.62%

The test accuracy is slightly lower than the validation accuracy, which is expected because the test set contains unseen data.

The small difference between the validation and test scores indicates that the model achieved relatively consistent performance on unseen data.


## Why Should the Test Set Not Be Used for Tuning?

The test set should only be used for the final evaluation.

If the test set were repeatedly used during model tuning, information from the test data would influence decisions such as model selection and hyperparameter choices. This could cause the model to become overfitted to the test set.

As a result, the test score would no longer provide an honest estimate of how the model performs on completely unseen data.

Therefore, the validation set was used for tuning, while the test set was kept untouched until the final evaluation.
